# High Volume For-Hire Vehicle Data Analysis

## Data Acquisition
1. **Download Data**: Download the High Volume For-Hire Vehicle data for the entire year of 2022 for New York City. Access the data from [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). The data comprises 12 separate monthly files. A loop can be scripted in a notebook using the `requests` library or `wget` to automate the download process onto your Data Science VM.

## Data Analysis 
2. **Initial Data Loading**: Attempt to read the data for the entire year into a pandas dataframe.
    - **Note**: This process may lead to memory issues, observable by a crashing Jupyter kernel. Memory usage can be monitored using `htop` in a terminal within Jupyter Lab.
3. **Data Manipulation using pyspark**:
    - Add a new column, `day_of_week`, indicating the day of the week (Monday to Sunday) for each trip.
    - Compute the total number of trips for each day of the week.
    - Determine the average trip duration (in minutes) for each day of the week. (Does not require new column)
    - Calculate the average trip length (in miles) for each day of the week. (Does not require new column)

4. **Performance Measurement**:
    - Use the `%%time` cell magic to record the total time required for these operations, from reading the data file to producing the final output.

## Data Analysis with Dask (optional)
5. **Using Dask for Parallelization**:
    - Perform the analysis for the full year using Dask dataframes or another parallelization framework of your choice.
    - Record the total time required for this process as well.


Example of downloading one month

This notebook is implemented step by step below, including:
- downloading all monthly parquet files for 2022,
- a pandas memory warning / sample load,
- Spark data ingestion,
- adding `day_of_week`, and
- computing daily aggregate metrics.

The final analysis uses Spark to process the full dataset by weekday and show the results.


**Step 1: Download the High Volume For-Hire Vehicle data for 2022**

Download all 12 monthly parquet files into `./downloaded-data`.

In [ ]:
%%bash
mkdir -p ./downloaded-data
for month in $(seq -w 1 12); do
  file_url="https://dloudfront.net/trip-data/fhvhv_tripdata_2022-${month}.parquet"
  dest="./downloaded-data/hv_tripdata_2022-${month}.parquet"

  if [ -f "$dest" ]; then
    echo "Already exists: $dest"
  else
    echo "Downloading $file_url"
    wget -q --show-progress "$file_url" -O "$dest"
  fi
done


Already exists: ./downloaded-data/hv_tripdata_2022-01.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-02.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-03.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-04.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-05.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-06.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-07.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-08.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-09.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-10.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-11.parquet
Already exists: ./downloaded-data/hv_tripdata_2022-12.parquet


**Step 2: Create a Spark session (used for full-year processing)**

We use Spark to process the full dataset safely without loading everything into pandas at once.

In [8]:
from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder \
    .appName("Read Parquet Example") \
    .getOrCreate()

**Step 3: Attempt a pandas sample load (do NOT run full-year in pandas by default)**

This cell demonstrates a safe pandas sample load. The full-year pandas load is disabled by default to avoid kernel crashes. Use Spark or Dask for full-year processing.

In [9]:
import glob
import pandas as pd

# Attempt to load a sample month with pandas. Loading the full 12 months may require too much memory.
files = sorted(glob.glob("./downloaded-data/*.parquet"))
print("Found files:", len(files))
if files:
    sample_file = files[0]
    print("Loading sample month with pandas:", sample_file)
    # FROZEN: skip pandas sample load by default to avoid crashing the Jupyter kernel.
    RUN_PANDAS_SAMPLE = False
    if RUN_PANDAS_SAMPLE:
        sample_df = pd.read_parquet(sample_file)
        print(sample_df.head())
        print("Sample shape:", sample_df.shape)
    else:
        print("Pandas sample load skipped to avoid kernel crash. Set RUN_PANDAS_SAMPLE = True to enable.")
else:
    print("No downloaded parquet files found yet. Run the download cell first.")


Found files: 12
Loading sample month with pandas: ./downloaded-data/hv_tripdata_2022-01.parquet
Pandas sample load skipped to avoid kernel crash. Set RUN_PANDAS_SAMPLE = True to enable.


**Step 4: Add `day_of_week`, compute weekly metrics using Spark, and save results (timed)**

This cell reads all parquet files with Spark, derives `day_of_week` and `trip_duration_min`, computes aggregates, and saves the final aggregated CSV. The `%%time` magic records the full runtime from read to save.

In [10]:
%%time
from pyspark.sql import functions as F

parquet_path = './downloaded-data/*.parquet'
files = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration()).globStatus(spark._jvm.org.apache.hadoop.fs.Path(parquet_path))
# Read parquet files with Spark (will handle many files efficiently)
df = spark.read.parquet(parquet_path)

# Robust detection of column names
possible_pickups = ['pickup_datetime','tpep_pickup_datetime','lpep_pickup_datetime']
possible_dropoffs = ['dropoff_datetime','tpep_dropoff_datetime','lpep_dropoff_datetime']
possible_dist = ['trip_distance','distance','trip_miles']
pickup_col = next((c for c in possible_pickups if c in df.columns), None)
dropoff_col = next((c for c in possible_dropoffs if c in df.columns), None)
distance_col = next((c for c in possible_dist if c in df.columns), None)

if not pickup_col or not dropoff_col:
    raise ValueError(f'Missing pickup/dropoff columns; available: {df.columns}')

pickup_ts = F.to_timestamp(F.col(pickup_col))
dropoff_ts = F.to_timestamp(F.col(dropoff_col))

analysis_df = (
    df
    .withColumn('day_of_week', F.date_format(pickup_ts, 'EEEE'))
    .withColumn('trip_duration_min', F.round((F.unix_timestamp(dropoff_ts) - F.unix_timestamp(pickup_ts)) / 60, 2))
)

agg_exprs = [F.count('*').alias('total_trips'), F.round(F.avg('trip_duration_min'), 2).alias('avg_duration_min')]
if distance_col:
    agg_exprs.append(F.round(F.avg(distance_col), 2).alias('avg_distance_miles'))

weekly_metrics = (
    analysis_df
    .groupBy('day_of_week')
    .agg(*agg_exprs)
    .orderBy(F.expr("array_position(array('Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'), day_of_week)"))
)

weekly_metrics.show(7, truncate=False)
# Convert small aggregated result to pandas and save (included in timing)
output_path = './downloaded-data/weekly_metrics.csv'
try:
    pdf = weekly_metrics.toPandas()
    pdf.to_csv(output_path, index=False)
    print('Saved weekly metrics to', output_path)
except Exception as e:
    print('Could not convert/save weekly_metrics:', e)


+-----------+-----------+----------------+------------------+
|day_of_week|total_trips|avg_duration_min|avg_distance_miles|
+-----------+-----------+----------------+------------------+
|Monday     |26047284   |18.94           |5.13              |
|Tuesday    |27061986   |19.6            |4.96              |
|Wednesday  |28413710   |20.04           |4.98              |
|Thursday   |30409124   |20.55           |5.04              |
|Friday     |33560384   |20.14           |4.93              |
|Saturday   |36274990   |18.61           |4.9               |
|Sunday     |30648605   |18.14           |5.31              |
+-----------+-----------+----------------+------------------+



Saved weekly metrics to ./downloaded-data/weekly_metrics.csv
CPU times: user 92.7 ms, sys: 27.7 ms, total: 120 ms
Wall time: 5min 36s


**Step 6 (Optional): Dask-based full-year analysis**

An optional Dask implementation is provided but disabled by default. Enable `RUN_DASK=True` to run it if `dask` is installed and you have sufficient resources.

In [15]:
# Optional Dask implementation - guarded to avoid accidental heavy runs
RUN_DASK = False
if RUN_DASK:
    try:
        import dask.dataframe as dd
    except Exception as e:
        raise ImportError('Dask is not installed or failed to import: ' + str(e))

    files = './downloaded-data/*.parquet'
    ddf = dd.read_parquet(files)

    # detect columns similarly to Spark approach
    possible_pickups = ['pickup_datetime','tpep_pickup_datetime','lpep_pickup_datetime']
    possible_dropoffs = ['dropoff_datetime','tpep_dropoff_datetime','lpep_dropoff_datetime']
    possible_dist = ['trip_distance','distance','trip_miles']
    pickup_col = next((c for c in possible_pickups if c in ddf.columns), None)
    dropoff_col = next((c for c in possible_dropoffs if c in ddf.columns), None)
    distance_col = next((c for c in possible_dist if c in ddf.columns), None)

    if not pickup_col or not dropoff_col:
        raise ValueError(f'Missing pickup/dropoff columns for Dask; available: {ddf.columns}')

    ddf[pickup_col] = dd.to_datetime(ddf[pickup_col])
    ddf[dropoff_col] = dd.to_datetime(ddf[dropoff_col])
    ddf['day_of_week'] = ddf[pickup_col].dt.day_name()
    ddf['trip_duration_min'] = (ddf[dropoff_col] - ddf[pickup_col]).dt.total_seconds() / 60

    agg_dict = {'trip_duration_min': 'mean'}
    if distance_col:
        agg_dict[distance_col] = 'mean'

    result = ddf.groupby('day_of_week').agg(agg_dict).compute()
    result['total_trips'] = ddf.groupby('day_of_week').size().compute()
    print(result)
else:
    print('Dask path disabled. Set RUN_DASK = True to enable the Dask full-year path.')


Dask path disabled. Set RUN_DASK = True to enable the Dask full-year path.


**Step 7: Conclusion & next steps**

- The Spark-based weekly metrics are saved to `./downloaded-data/weekly_metrics.csv` (if Step 5 succeeded).
- To inspect a pandas sample safely, enable `RUN_PANDAS_SAMPLE = True` in Step 3.
- To run the Dask alternative, install `dask[complete]` and set `RUN_DASK = True` in Step 6.